In [42]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen2.5:7b", think = False)

llm

ChatOllama(model='qwen2.5:7b')

Parser를 이용해 Output 중 원하는 것만 파싱.

In [45]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
prompt_template = PromptTemplate(
    template="""
    What is the Capital of {country}? Return only the name of the city
    """,
    input_variables=["country"],
)

output_parser = StrOutputParser()

answer = output_parser.invoke(llm.invoke(prompt_template.invoke({"country" : "Korea"})))

answer

'Seoul'

위의 방식을 Chain으로 연결도 가능.
Runnable 들은 파일프(chain)으로 연결 가능.
chain도 Ruannable임. 또 다른 거에 연결 가능.

In [46]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
prompt_template = PromptTemplate(
    template="""
    What is the Capital of {country}? Return only the name of the city
    """,
    input_variables=["country"],
)

chain = prompt_template | llm | StrOutputParser()

answer = chain.invoke({"country" : "Korea"})

answer

'Seoul'

특정 JSON 형식으로 리턴하도록 할 수 있다. BaseModel은 LLM응답을 담는 DTO 역할, @Valid, @Notnull 같은 기능도 있다.
Field(description) 은 LLM이 각 필드에 뭘 넣어야 할지 힌트를 주는 용도

In [49]:
from pydantic import BaseModel, Field

class CountryDetail(BaseModel):
    capital: str = Field(description = "The capital of the country")
    population: int = Field(description = "The population of the country")
    language: str = Field(description = "The language of the country")
    currency: str = Field(description = "The currency of the country")


In [61]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    template="""
    Give following information about {country} :
    - Capital
    - Population
    - Language
    - Currency

    return it in JSON format. and return the JSON dictionary only
    """,
    input_variables=["country"],
)

CountryInfoChain = prompt_template | llm.with_structured_output(CountryDetail)

answer = CountryInfoChain.invoke({"country" : "Korea"})

answer
answer.model_dump()

{'capital': 'Seoul',
 'population': 51807300,
 'language': 'Korean',
 'currency': 'South Korean won'}

In [62]:
from langchain_core.prompts import PromptTemplate

country_prompt = PromptTemplate(
    template="""
    Guess the name of the country based on the following information:
    {information}

    return only the name of the country
    """,
    input_variables=["information"],
)

countryNameChain = country_prompt | llm | output_parser

answer = countryNameChain.invoke({"information" : "The laragest country of the world"})

answer

'Russia'

In [66]:
finalChain = countryNameChain | CountryInfoChain

answer = finalChain.invoke({"information" : "The smallest country of the wourld" })

answer.model_dump()

{'capital': 'Vatican City',
 'population': 842,
 'language': 'Italian, Latin',
 'currency': 'Euro (EUR)'}